# 03 — Layout-Aware Semantic Parsing (Milestone M2)

**DSML stage:** data preparation. **Input:** raw filing HTML from notebook 01. **Output:** per-filing
section-segmented element tables in `data/interim/sections/`.

`sec-parser` classifies every visual element of an EDGAR filing (`TextElement`, `TitleElement`,
`TableElement`, plus noise classes like `PageHeaderElement`). We then segment the element stream into
**SEC Items** (Item 1 Business, Item 1A Risk Factors, Item 7 MD&A, ...) — these become `FilingSection`
nodes in the graph.

> **Library caveat (documented finding):** sec-parser 0.58 ships only `Edgar10QParser`; on 10-Ks it emits
> *invalid section type* warnings and misses some `TopSectionTitle`s (e.g. Item 1A arrives as a plain
> `TitleElement`). We therefore do our own **part-aware regex segmentation** over the classified elements —
> robust for both 10-K and 10-Q (where item numbers repeat across Part I/II).

In [1]:
import json
import re
import warnings
from pathlib import Path

import pandas as pd
import sec_parser as sp

warnings.filterwarnings("ignore", category=UserWarning)  # sec-parser 10-K section-type noise

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MANIFEST = json.loads((PROJECT_ROOT / "data/raw/edgar/NVDA/manifest.json").read_text())
OUT_DIR = PROJECT_ROOT / "data" / "interim" / "sections"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"{len(MANIFEST)} filings to parse")
pd.DataFrame(MANIFEST)[["form", "filing_date", "accession_no"]]

5 filings to parse


,form,filing_date,accession_no
0,10-K,2026-02-25,0001045810-26-000021
1,10-K,2025-02-26,0001045810-25-000023
2,10-K,2024-02-21,0001045810-24-000029
3,10-K,2023-02-24,0001045810-23-000017
4,10-Q,2026-05-20,0001045810-26-000052


## 1. Part-aware section segmentation

A heading starts a new section when it is a short `TitleElement`/`TopSectionTitle` matching `Item N.` or
`Part N`. Section ids are qualified by part (`I.1A`, `II.1A`) so 10-Q item-number collisions stay distinct.
Noise elements are dropped; every kept element records its type so the chunker (notebook 04) can treat
tables specially.

In [2]:
ITEM_RE = re.compile(r"^item\s+(\d+[a-z]?)[\.\:\s]", re.IGNORECASE)
# Matches "Part II", "PART II — OTHER INFORMATION", "Part I." etc. (prefix match, roman numeral captured)
PART_RE = re.compile(r"^part\s+(i{1,3}|iv)\b", re.IGNORECASE)
NOISE_TYPES = {"IrrelevantElement", "PageHeaderElement", "PageNumberElement", "EmptyElement",
               "NotYetClassifiedElement", "ImageElement", "IntroductorySectionElement"}
HEADING_TYPES = {"TitleElement", "TopSectionTitle"}

def segment_filing(html: str) -> pd.DataFrame:
    """Parse filing HTML and return one row per content element, tagged with its SEC section."""
    elements = sp.Edgar10QParser().parse(html)
    part, section_id, section_title = "I", None, None
    rows = []
    for idx, el in enumerate(elements):
        el_type = type(el).__name__
        text = (el.text or "").strip()
        if not text or el_type in NOISE_TYPES:
            continue
        if el_type in HEADING_TYPES and len(text) < 200:
            pm = PART_RE.match(text)
            if pm:
                part = pm.group(1).upper()
                continue
            m = ITEM_RE.match(text)
            if m:
                section_id = f"{part}.{m.group(1).upper()}"
                section_title = text
                continue
        if section_id is None:
            continue  # cover page / TOC before the first item
        rows.append({
            "element_index": idx,
            "section_id": section_id,
            "section_title": section_title,
            "element_type": el_type,
            "text": text,
        })
    return pd.DataFrame(rows)

In [3]:
all_sections = {}
for m in MANIFEST:
    html = (PROJECT_ROOT / m["local_path"]).read_text(encoding="utf-8")
    df = segment_filing(html)
    df.insert(0, "accession_no", m["accession_no"])
    df.insert(1, "form", m["form"])
    df.insert(2, "filing_date", m["filing_date"])
    out_path = OUT_DIR / f"{m['accession_no']}.parquet"
    df.to_parquet(out_path, index=False)
    all_sections[m["accession_no"]] = df
    print(f"{m['form']} {m['filing_date']}: {len(df):4d} elements, {df['section_id'].nunique():2d} sections → {out_path.name}")

10-K 2026-02-25:  506 elements, 22 sections → 0001045810-26-000021.parquet


10-K 2025-02-26:  531 elements, 22 sections → 0001045810-25-000023.parquet


10-K 2024-02-21:  523 elements, 22 sections → 0001045810-24-000029.parquet


10-K 2023-02-24:  542 elements, 21 sections → 0001045810-23-000017.parquet
10-Q 2026-05-20:  265 elements,  9 sections → 0001045810-26-000052.parquet


## 2. Inspect the extraction targets — Risk Factors (1A) and MD&A (7 / 10-Q II.2)

In [4]:
combined = pd.concat(all_sections.values(), ignore_index=True)
combined["n_words"] = combined["text"].str.split().str.len()

summary = (
    combined.groupby(["form", "filing_date", "section_id"])
    .agg(elements=("text", "count"), words=("n_words", "sum"), tables=("element_type", lambda s: (s == "TableElement").sum()))
    .reset_index()
)
focus = summary[summary["section_id"].isin(["I.1", "I.1A", "II.7", "I.2", "II.1A"])]
focus.sort_values(["filing_date", "section_id"])

,form,filing_date,section_id,elements,words,tables
0,10-K,2023-02-24,I.1,54,7586,1
1,10-K,2023-02-24,I.1A,65,12985,0
3,10-K,2023-02-24,I.2,1,137,0
7,10-K,2023-02-24,II.7,78,5688,8
21,10-K,2024-02-21,I.1,50,7382,1
22,10-K,2024-02-21,I.1A,63,15841,0
25,10-K,2024-02-21,I.2,1,137,0
29,10-K,2024-02-21,II.7,77,5960,9
43,10-K,2025-02-26,I.1,50,7862,1
44,10-K,2025-02-26,I.1A,62,16385,0


In [5]:
# Eyeball a risk-factor paragraph from the latest 10-K — this is what the LLM will extract from.
latest_10k = [m for m in MANIFEST if m["form"] == "10-K"][0]["accession_no"]
risk_1a = all_sections[latest_10k].query("section_id == 'I.1A' and element_type == 'TextElement'")
supply_paras = risk_1a[risk_1a["text"].str.contains("supply|foundr|manufactur", case=False)]
print(f"Item 1A: {len(risk_1a)} text elements, {len(supply_paras)} mention supply/foundry/manufacturing\n")
print(supply_paras.iloc[0]["text"][:600], "...")

Item 1A: 28 text elements, 14 mention supply/foundry/manufacturing

•Long manufacturing lead times and uncertain supply and capacity availability, combined with a failure to estimate customer demand accurately has led and could lead to mismatches between supply and demand.•Dependency on third-party suppliers and their technology to manufacture, assemble, test, or package our products reduces our control over product quantity and quality, manufacturing yields, and product delivery schedules and could harm our business.•Defects in our products have caused and could cause us to incur significant expenses to remediate and could damage our business. ...


## 3. Data dictionary — `data/interim/sections/{accession_no}.parquet`

| column | meaning |
|---|---|
| `accession_no` | EDGAR filing id (joins to manifest + future `Filing` node) |
| `form`, `filing_date` | filing metadata denormalized for convenience |
| `element_index` | position in sec-parser's element stream (stable within a filing → provenance) |
| `section_id` | part-qualified item id: `I.1A` = Part I Item 1A (10-K Risk Factors), `II.7` = MD&A, 10-Q risk factors = `II.1A` |
| `section_title` | the heading text as printed |
| `element_type` | `TextElement` \| `TableElement` \| `TitleElement` (sub-headings) \| `SupplementaryText` |
| `text` | cleaned element text (tables flattened to linear text) |

**Known limitation:** sec-parser flattens table structure to linear text. Fine for the PoC (financial
*numbers* come from XBRL, not tables); revisit with a vision parser (Docling/LlamaParse) if table
semantics are ever needed for extraction.

In [6]:
# --- M2 assertion cell ---
for m in MANIFEST:
    df = all_sections[m["accession_no"]]
    sections = set(df["section_id"])
    if m["form"] == "10-K":
        assert "I.1A" in sections and "II.7" in sections, f"{m['accession_no']}: missing 1A or 7 — got {sorted(sections)}"
        risk_words = df.query("section_id == 'I.1A'")["text"].str.split().str.len().sum()
        assert risk_words > 10_000, f"{m['accession_no']}: Item 1A suspiciously short ({risk_words} words)"
    else:  # 10-Q
        assert "II.1A" in sections and "I.2" in sections, f"{m['accession_no']}: missing 10-Q risk/MD&A — got {sorted(sections)}"
assert (OUT_DIR / f"{MANIFEST[0]['accession_no']}.parquet").exists()
print(f"M2 (parsing) OK — {len(MANIFEST)} filings segmented, Risk Factors + MD&A present in all")

M2 (parsing) OK — 5 filings segmented, Risk Factors + MD&A present in all
